# Task 4: Sentiment Analysis Using NLP

**Objective:** Perform sentiment analysis on textual data (customer reviews) using Natural Language Processing (NLP).

This notebook demonstrates data exploration, text preprocessing, TF-IDF feature extraction, Logistic Regression model training, evaluation, predictions, and insights.

> **Dataset note:** A small example dataset is included so the notebook runs immediately. Its results are for learning and demonstration only. Replace it with a larger real-world labelled dataset for a stronger project.

## 1. Import libraries

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print("Libraries imported successfully.")

## 2. Create or load the dataset

The sample contains customer reviews labelled `positive` or `negative`. To use your own CSV, see the optional section at the end.

In [ ]:
reviews = [('The product is excellent and works perfectly', 'positive'), ('Amazing quality, I am very happy with this purchase', 'positive'), ('Fast delivery and great customer service', 'positive'), ('I love this product, it is worth the money', 'positive'), ('Very useful and easy to use', 'positive'), ('The experience was fantastic and wonderful', 'positive'), ('Good quality and quick delivery', 'positive'), ('Highly recommended, completely satisfied', 'positive'), ('The app is simple, smooth and helpful', 'positive'), ('Excellent service, friendly staff', 'positive'), ('This is the best purchase I have made', 'positive'), ('Beautiful design and excellent performance', 'positive'), ('The product exceeded my expectations', 'positive'), ('Very pleased with the quality', 'positive'), ('Works well and arrived on time', 'positive'), ('Great value for money', 'positive'), ('I am satisfied and would buy again', 'positive'), ('The support team solved my issue quickly', 'positive'), ('Fantastic experience from start to finish', 'positive'), ('Reliable product with impressive features', 'positive'), ('The product is terrible and stopped working', 'negative'), ('Very poor quality, I regret buying this', 'negative'), ('Delivery was late and the package was damaged', 'negative'), ('I am disappointed with this purchase', 'negative'), ('The app crashes and is difficult to use', 'negative'), ('Waste of money, do not recommend it', 'negative'), ('Customer service was rude and unhelpful', 'negative'), ('The product does not work properly', 'negative'), ('Bad experience, very frustrating', 'negative'), ('Poor performance and disappointing quality', 'negative'), ('I hate this product, it is useless', 'negative'), ('The order arrived broken', 'negative'), ('Not worth the price at all', 'negative'), ('The service was slow and careless', 'negative'), ('I will never buy this again', 'negative'), ('The instructions are confusing and unclear', 'negative'), ('Terrible experience and no support', 'negative'), ('The product feels cheap and unreliable', 'negative'), ('It failed after only one day', 'negative'), ('Extremely unhappy with the service', 'negative')]
df = pd.DataFrame(reviews, columns=['text', 'sentiment'])
df.head()

## 3. Explore the data

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nClass distribution:\n", df["sentiment"].value_counts())

df["sentiment"].value_counts().plot(kind="bar", title="Number of reviews by sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of reviews")
plt.xticks(rotation=0)
plt.show()

## 4. Text preprocessing

The cleaning function converts text to lowercase, removes URLs, mentions, hashtags, punctuation, numbers and extra spaces.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text", "sentiment"]].head(10)

## 5. Split the data and build the model

TF-IDF converts text into numerical features. Logistic Regression learns patterns that distinguish positive and negative reviews. The pipeline keeps vectorization and classification together.

In [ ]:
X = df["clean_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
print("Model training completed.")

## 6. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=["negative", "positive"])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative", "positive"]).plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## 7. Predict sentiment for new reviews

In [ ]:
new_reviews = [
    "The delivery was quick and the product is excellent",
    "Very disappointing quality, I want a refund",
    "The app is easy to use and works smoothly"
]

new_clean = [clean_text(review) for review in new_reviews]
predictions = model.predict(new_clean)

for review, sentiment in zip(new_reviews, predictions):
    print(f"Review: {review}\nPredicted sentiment: {sentiment}\n")

## 8. Insights and conclusion

After running the notebook, record:
- The number of positive and negative reviews.
- Test accuracy and class-wise precision, recall and F1-score.
- What the confusion matrix says about correct and incorrect predictions.
- Words or phrases that may influence predictions.
- Why a small manually-created dataset can produce misleading or unstable scores.

**Conclusion:** This project demonstrates an end-to-end NLP workflow using text cleaning, TF-IDF and Logistic Regression. For reliable conclusions, use thousands of real labelled reviews and evaluate on a separate test set.

## 9. Optional: Use your own CSV dataset

Prepare a CSV file with columns named `text` and `sentiment`, then replace the sample-data cell with:

```python
df = pd.read_csv("your_reviews.csv")
df = df[["text", "sentiment"]].dropna()
df["sentiment"] = df["sentiment"].str.lower().str.strip()
```

For a three-class task, include `neutral` labels and ensure each class has enough examples.